In [0]:
storage_account = "sreyaastorage"

client_id = dbutils.secrets.get(
    scope="sreyascope",
    key="clientid"
)

client_secret = dbutils.secrets.get(
    scope="sreyascope",
    key="clientsecretvalue"
)

tenant_id = dbutils.secrets.get(
    scope="sreyascope",
    key="tenantsecretid"
)

In [0]:
bronze_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
)

silver_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
)

In [0]:

storage_account = "sreyaastorage"
host = f"{storage_account}.dfs.core.windows.net"

client_id = dbutils.secrets.get(
    scope="sreyascope",
    key="clientid"
)

client_secret = dbutils.secrets.get(
    scope="sreyascope",
    key="clientsecretvalue"
)

tenant_id = dbutils.secrets.get(
    scope="sreyascope",
    key="tenantsecretid"
)

spark.conf.set(
    f"fs.azure.account.auth.type.{host}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{host}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{host}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{host}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{host}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("OAuth configuration completed")


OAuth configuration completed


In [0]:

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"

transactions_df = spark.read.parquet(
    f"{bronze_path}transactions/"
)

display(transactions_df)


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


In [0]:

storage_account = "sreyaastorage"

bronze_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
)

transactions_df = spark.read.parquet(
    f"{bronze_path}transactions/"
)

products_df = spark.read.parquet(
    f"{bronze_path}products/"
)

stores_df = spark.read.parquet(
    f"{bronze_path}stores/"
)

customers_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{bronze_path}customers/customers.json")
)

print("All Bronze datasets loaded successfully!")


All Bronze datasets loaded successfully!


In [0]:

from pyspark.sql.functions import col, to_date

# Remove duplicate records
transactions_silver = transactions_df.dropDuplicates()
products_silver = products_df.dropDuplicates()
stores_silver = stores_df.dropDuplicates()
customers_silver = customers_df.dropDuplicates()

# Convert date columns into proper date format
transactions_silver = transactions_silver.withColumn(
    "transaction_date",
    to_date(col("transaction_date"))
)

customers_silver = customers_silver.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

# Ensure product prices are numeric
products_silver = products_silver.withColumn(
    "price",
    col("price").cast("decimal(10,2)")
)

# Display the cleaned data
display(transactions_silver)
display(products_silver)
display(stores_silver)
display(customers_silver)


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
6,110,7,3,5,2025-01-04
19,116,8,4,4,2024-07-14
16,118,6,2,4,2024-11-29
17,109,8,5,5,2024-07-10
12,108,5,1,4,2025-05-26
5,105,5,2,1,2025-03-17
22,107,9,3,4,2024-11-16
24,125,1,5,1,2024-07-14
11,102,1,3,2,2024-08-11
20,130,5,1,2,2024-07-30


product_id,product_name,category,price
7,Smartwatch,Electronics,4999.00
6,Water Bottle,Fitness,299.00
2,Bluetooth Speaker,Electronics,1299.49
1,Wireless Mouse,Electronics,799.99
3,Yoga Mat,Fitness,499.00
9,Dumbbell Set,Fitness,1999.00
5,Notebook Set,Stationery,149.00
4,Laptop Stand,Accessories,999.99
10,Pen Drive 32GB,Electronics,599.00
8,Desk Organizer,Accessories,399.00


store_id,store_name,location
3,Tech World Outlet,Bangalore
1,City Mall Store,Mumbai
5,Mega Plaza,Chennai
2,High Street Store,Delhi
4,Downtown Mini Store,Pune


city,customer_id,email,first_name,last_name,phone,registration_date
Mumbai,102,user102@example.com,Nina,Joshi,9876543210,2024-01-21
Delhi,117,user117@example.com,Neha,Verma,9721098765,2024-03-01
Chennai,137,user137@example.com,Shweta,Mehta,9521098765,2023-05-06
Bangalore,143,user143@example.com,Payal,Sharma,9465432109,2023-08-16
Kolkata,124,user124@example.com,Kavita,Sharma,9654321098,2023-11-15
Hyderabad,104,user104@example.com,Karan,Patel,9854321098,2024-02-05
Chennai,113,user113@example.com,Divya,Singh,9765432109,2024-02-18
Ahmedabad,123,user123@example.com,Arjun,Yadav,9665432109,2024-02-25
Hyderabad,128,user128@example.com,Simran,Joshi,9610987654,2023-06-11
Delhi,141,user141@example.com,Rachna,Nair,9487654321,2024-02-27


In [0]:
print("Transactions:", transactions_df.count())
print("Products:", products_df.count())
print("Stores:", stores_df.count())
print("Customers:", customers_df.count())

Transactions: 30
Products: 10
Stores: 5
Customers: 50


In [0]:

from pyspark.sql.functions import col, to_date

# 1. Remove duplicate rows
transactions_silver = transactions_df.dropDuplicates()
products_silver = products_df.dropDuplicates()
stores_silver = stores_df.dropDuplicates()
customers_silver = customers_df.dropDuplicates()

# 2. Convert transaction date
transactions_silver = transactions_silver.withColumn(
    "transaction_date",
    to_date(col("transaction_date"))
)

# 3. Convert customer registration date
customers_silver = customers_silver.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

# 4. Convert product price to decimal
products_silver = products_silver.withColumn(
    "price",
    col("price").cast("decimal(10,2)")
)

# 5. Display the cleaned datasets
display(transactions_silver)
display(products_silver)
display(stores_silver)
display(customers_silver)


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
6,110,7,3,5,2025-01-04
19,116,8,4,4,2024-07-14
16,118,6,2,4,2024-11-29
17,109,8,5,5,2024-07-10
12,108,5,1,4,2025-05-26
5,105,5,2,1,2025-03-17
22,107,9,3,4,2024-11-16
24,125,1,5,1,2024-07-14
11,102,1,3,2,2024-08-11
20,130,5,1,2,2024-07-30


product_id,product_name,category,price
7,Smartwatch,Electronics,4999.00
6,Water Bottle,Fitness,299.00
2,Bluetooth Speaker,Electronics,1299.49
1,Wireless Mouse,Electronics,799.99
3,Yoga Mat,Fitness,499.00
9,Dumbbell Set,Fitness,1999.00
5,Notebook Set,Stationery,149.00
4,Laptop Stand,Accessories,999.99
10,Pen Drive 32GB,Electronics,599.00
8,Desk Organizer,Accessories,399.00


store_id,store_name,location
3,Tech World Outlet,Bangalore
1,City Mall Store,Mumbai
5,Mega Plaza,Chennai
2,High Street Store,Delhi
4,Downtown Mini Store,Pune


city,customer_id,email,first_name,last_name,phone,registration_date
Mumbai,102,user102@example.com,Nina,Joshi,9876543210,2024-01-21
Delhi,117,user117@example.com,Neha,Verma,9721098765,2024-03-01
Chennai,137,user137@example.com,Shweta,Mehta,9521098765,2023-05-06
Bangalore,143,user143@example.com,Payal,Sharma,9465432109,2023-08-16
Kolkata,124,user124@example.com,Kavita,Sharma,9654321098,2023-11-15
Hyderabad,104,user104@example.com,Karan,Patel,9854321098,2024-02-05
Chennai,113,user113@example.com,Divya,Singh,9765432109,2024-02-18
Ahmedabad,123,user123@example.com,Arjun,Yadav,9665432109,2024-02-25
Hyderabad,128,user128@example.com,Simran,Joshi,9610987654,2023-06-11
Delhi,141,user141@example.com,Rachna,Nair,9487654321,2024-02-27


In [0]:

spark.sql("""
CREATE SCHEMA IF NOT EXISTS sreyadatabricks.silver
""")

print("Silver schema is ready!")


Silver schema is ready!


In [0]:

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

transactions_silver.write.mode("overwrite").parquet(
    f"{silver_path}transactions/"
)

products_silver.write.mode("overwrite").parquet(
    f"{silver_path}products/"
)

stores_silver.write.mode("overwrite").parquet(
    f"{silver_path}stores/"
)

customers_silver.write.mode("overwrite").parquet(
    f"{silver_path}customers/"
)

print("Silver layer created successfully!")


Silver layer created successfully!
